# 00 · Business Context and Project Charter

**Meridian Capital · Counterparty Risk Intelligence Platform**

| | |
|---|---|
| **Notebook purpose** | Define the business problem, the decisions this project supports, how success is measured and how the work is organised. |
| **Audience** | Hiring managers, risk leaders, data and analytics engineers, non-technical stakeholders. |
| **Reading time** | 6 minutes |

---

## 1. The company

**Meridian Capital** is a fictional mid-size lender used as the setting for this project. All data is real and public. The company and its numbers are a scenario built on top of that data.

Meridian runs three businesses:

| Business line | What it does | Data used as its book |
|---|---|---|
| **Consumer and small-business lending** | Unsecured term loans of 36 and 60 months, USD 1,000 to 40,000. | Lending Club, 2.26 million loans, 2007 to 2018 |
| **Mobile payments** | Transfers, cash-in, cash-out and merchant payments through a mobile wallet. | PaySim, 6.36 million transactions modelled on African mobile-money logs |
| **Supplier finance and procurement advisory** | Pays corporate clients' suppliers early and advises procurement teams on supplier risk. | SEC Financial Statement Data Sets, 4,400 US companies' FY2018 annual reports |

## 2. The problem

The board has raised three concerns at the same quarterly risk committee:

1. **Credit losses are rising.** The 24-month bad rate on new loans climbed from 8.3% (2013) to 11.4% (2016), during the strongest economy in a decade. The Chief Risk Officer wants to know who will default, how much Meridian will lose and what a recession would do to the book.
2. **Fraud is slipping through.** The current rule engine on the payments channel catches almost nothing. The Head of Fraud needs a model that pays for itself.
3. **Corporate clients want supplier risk scores.** Procurement teams at Meridian's clients ask which of their suppliers could fail, and how much of their spend is exposed.

All three share one need: **a reliable, explainable estimate of counterparty risk turned into money.**

## 3. Decisions this project supports

| Stakeholder | Decision | Product delivered |
|---|---|---|
| Chief Risk Officer | Approve, decline or price a loan application | Probability of Default (PD) scorecard and challenger model with reason codes |
| Chief Financial Officer | Set loss provisions and capital buffers | Expected Loss = PD x LGD x EAD, macro stress test |
| Head of Fraud | Which transactions to hold for review | Fraud model with a cost-optimised alert threshold |
| Procurement (client side) | Which suppliers to dual-source, monitor or offer early payment | Supplier financial health score and spend-at-risk view |
| Model Risk Management | Is the model still fit for use | Validation pack, drift monitoring (PSI), model card |
| Everyone | Ask questions without writing SQL | AI Risk Analyst agent over the governed warehouse |

## 4. How success is measured

### 4.1 KPI tree

Every chart in the dashboard maps to one branch of this tree. The top line is the number the board cares about. Each level below it is something a team can act on.

```
Risk-adjusted return on the loan book
├── Net interest income
│   ├── Funded volume           (loans originated, USD funded, approval rate)
│   └── Yield                   (weighted interest rate by grade)
└── Credit losses
    ├── Expected Loss           = PD x LGD x EAD
    │   ├── PD                  (model AUC / Gini / KS, calibration gap)
    │   ├── LGD                 (recovery rate on defaults)
    │   └── EAD                 (balance outstanding at default)
    ├── Portfolio quality       (vintage curves, grade mix, concentration HHI)
    └── Stress loss             (EL under adverse and severe scenarios)

Fraud losses avoided
├── Detection rate (recall)     share of fraud value caught
├── Precision                   share of alerts that are real fraud
└── Review cost                 alerts per 1,000 transactions x cost per review

Supplier resilience (client procurement)
├── Spend at risk               spend with suppliers in the distress zone
├── Concentration               share of spend with top suppliers
└── Coverage                    share of spend with a current health score
```

### 4.2 ROI framework

A model is only worth deploying if it changes a decision for the better. Each model is evaluated against the policy Meridian uses today, in dollars:

| Model | Baseline (today) | How value is measured |
|---|---|---|
| PD scorecard and challenger | Approve every application that meets Lending Club's own policy | Profit per USD 1 billion originated: interest earned on good loans minus losses on bad loans, at the best cutoff |
| Fraud model | Existing rule engine (`isFlaggedFraud`) | Fraud value prevented minus the cost of reviewing alerts |
| Supplier score | No screening | Spend moved out of the distress zone, and supplier-finance revenue from grey-zone suppliers |

Every assumption (funding cost, cost per fraud review, recovery haircut) is stated next to its result and is adjustable in the dashboard's ROI calculator.

## 5. Data

| Source | Rows | Period | Licence | Role |
|---|---|---|---|---|
| [Lending Club accepted loans](https://huggingface.co/datasets/codesignal/lending-club-loan-accepted) | 2,260,701 | Jun 2007 to Dec 2018 | Public | Loan book: PD, LGD, EAD, vintages, ROI |
| [FRED](https://fred.stlouisfed.org/) (6 series) | 13,896 | 1947 to 2026 | Public domain | Macro drivers and stress scenarios |
| [SEC Financial Statement Data Sets](https://www.sec.gov/dera/data/financial-statement-data-sets) | 5.9M values, 12,940 filings | Filed H1 2019 | Public domain | Supplier financial health |
| [PaySim](https://huggingface.co/datasets/theman10/paysim) | 6,362,620 | 30 simulated days | CC BY-SA 4.0 | Payments fraud |

`scripts/download_data.py` fetches all four. Raw data is not committed to the repository.

## 6. Architecture

The warehouse follows a **medallion** design. Each layer has one job, and every gold table is rebuilt and tested by dbt on every run.

```
 Raw files            Bronze (Parquet)        Silver (dbt)                 Gold (dbt)                  Consumers
 ─────────            ────────────────        ────────────                 ──────────                  ─────────
 CSV / TSV   ──────►  typed, compressed  ──►  staging: clean, rename  ──►  star schema + marts   ──►  Notebooks
 (3.2 GB)             unchanged content       intermediate: business       fct_loans, dim_*,           Streamlit dashboard
                      (0.5 GB)                rules (default, LGD,         mart_pd_features,           AI Risk Analyst
                                              split) in one place          mart_vintage_curves ...
```

| Layer | Tool | Rule |
|---|---|---|
| Bronze | DuckDB `COPY ... TO parquet` | Never change content. Only change format. |
| Silver | dbt views and tables | Technical cleaning only (types, names, units) in staging. Business definitions only in intermediate. |
| Gold | dbt tables | Star schema and purpose-built marts. 43 automated data tests, including a leakage test. |

## 7. Notebook map

| # | Notebook | Question it answers |
|---|---|---|
| 00 | Business context | What problem are we solving and how do we measure success? |
| 01 | Data ingestion and SQL warehouse | Is the data loaded completely, and what does SQL already tell us? |
| 02 | Data audit | Can we trust the data? What is wrong with it? |
| 03 | Data cleaning | How do we fix each issue without leaking the future? |
| 04 | Exploratory analysis | What drives default? |
| 05 | Feature engineering and WoE | Which variables carry signal, and how do we encode them? |
| 06 | PD scorecard | A transparent, bank-standard default model |
| 07 | Challenger model and SHAP | Can machine learning beat it, and can we still explain it? |
| 08 | LGD, EAD and Expected Loss | How much will Meridian lose? |
| 09 | Stress testing and ROI | What if a recession hits, and what is the model worth? |
| 10 | Fraud detection | How much fraud can we stop, net of review cost? |
| 11 | Supplier risk | Which suppliers could fail, and how much spend is exposed? |
| 12 | Validation and monitoring | Is the model still fit for use? |
| 13 | AI Risk Analyst | Can a stakeholder ask the warehouse questions in plain English? |

## 8. Warehouse inventory

The cell below confirms that the warehouse described above exists and is populated.

In [1]:
import pandas as pd
from meridian.db import connect

con = connect()
inventory = con.execute("""
    select t.table_schema as layer, t.table_name, t.table_type,
           c.n_columns
    from information_schema.tables t
    join (select table_schema, table_name, count(*) as n_columns
          from information_schema.columns group by all) c using (table_schema, table_name)
    where t.table_schema in ('silver', 'gold', 'reference')
    order by case t.table_schema when 'silver' then 1 when 'gold' then 2 else 3 end, t.table_name
""").df()

inventory["rows"] = [
    con.execute(f'select count(*) from {s}.{t}').fetchone()[0]
    for s, t in zip(inventory.layer, inventory.table_name)
]
inventory.style.format({"rows": "{:,.0f}"}).hide(axis="index")

layer,table_name,table_type,n_columns,rows
silver,int_loans__outcomes,BASE TABLE,83,"2,260,668"
silver,int_sec__supplier_financials,BASE TABLE,24,"4,482"
silver,stg_fred__series,VIEW,4,"13,618"
silver,stg_lending_club__loans,BASE TABLE,69,"2,260,668"
silver,stg_paysim__transactions,BASE TABLE,15,"6,362,620"
silver,stg_sec__financial_values,BASE TABLE,5,"275,736"
silver,stg_sec__submissions,VIEW,11,"4,519"
gold,dim_geography,BASE TABLE,4,51
gold,dim_month,BASE TABLE,6,156
gold,dim_risk_grade,BASE TABLE,3,35


## 9. Scope, assumptions and limitations

- **Meridian is fictional.** Lending Club, PaySim and SEC filers stand in for Meridian's own books. Results are realistic in shape and order of magnitude, not a statement about any real company.
- **US data.** Lending Club and SEC data are US-only. The methods (scorecards, Expected Loss, stress testing, Altman Z) are the same ones used by banks in Europe, Latin America and Africa.
- **Accepted loans only.** Lending Club publishes only funded loans, so the PD model learns from approved applicants. This is a known limitation (reject inference) and is discussed in notebook 06.
- **PaySim is simulated.** It reproduces the statistical patterns of a real mobile-money service, including some simulator artefacts that make fraud easier to detect than in real life. Notebook 10 measures the model both with and without the features affected.
- **No protected attributes.** The data contains no race, sex or age, so fairness can only be checked on proxies. This is stated in the model card.

**Next:** notebook 01 loads the data and uses SQL to answer the first business questions.